In [1]:
# Import required libraries
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

**⚠️ Data Requirement — BioImageArchive:** This notebook requires raw microscopy data from BioImageArchive. Download the dataset and set `data_archive_path` to your local BioImageArchive directory (see README).

**Pipeline step 1/1** — outputs `hqno_calibration_microscopy.csv` and `rhl_calibration_microscopy.csv` used by `figure_code/1_plot_3A_callibration.ipynb`.

# Aggregate Fluorescence Intensity Data for HQNO and RHL reporter calibration  
## Overview

This notebook aggregates single-cell property data from multiple experimental replicates and positions for the SA HQNO and RHL reporter in co-culture with PA mutant deficient in HQNO or RHL production, respectively, exposed to exogenous HQNO and RHL

## Workflow
1. **Input**: Data comes from  `SAReporterCalibration` study component in BioImageArchive repo
2. **Data Collection**: Iterates over all replicates and positions and loads `single_cell_props.csv` files.
3. **Output**: Saves the combined dataframe as `hqno_calibration_microscopy.csv` and `rhl_calibration_microscopy.csv`

## Note
Code for further processing of data, model fitting, and figure plotting is located in accompanying [Github repository](https://github.com/simonvanvliet/SpatialToleranceModel)

In [2]:
# Define input and output paths
# set path to BioImageArchive data directory:
data_archive_path = Path('/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2026/BioImageArchive/')

# relative paths to data (wildcards to match all concentration folders)
input_dir_h =  data_archive_path / 'SAReporterCalibration' / 'S12_DpqsL_HQNO*'  
output_dir_h = 'hqno_calibration_microscopy.csv'

input_dir_r = data_archive_path / 'SAReporterCalibration' / 'S12_DrhlA_RHL*'
output_dir_r = 'rhl_calibration_microscopy.csv'

process_list = [
    (input_dir_h, output_dir_h, 'HQNO'),
    (input_dir_r, output_dir_r, 'RHL'),
]

In [3]:
for input_dir_pattern, output_dir, col_name in process_list:

    all_data = []  # List to store individual dataframes

    # Find all concentration folders matching the wildcard pattern
    conc_folders = sorted(glob.glob(str(input_dir_pattern)))

    if not conc_folders:
        raise FileNotFoundError(
            f"Could not find any directories matching: {input_dir_pattern}"
        )

    # Loop over each concentration folder (e.g. S12_DpqsL_HQNO-0ng, S12_DpqsL_HQNO-10ng, ...)
    for conc_folder in conc_folders:
        # Extract concentration label: last part of folder name, e.g. 'HQNO-0ng'
        conc_label = os.path.basename(conc_folder).split('_')[-1]

        # Iterate through replicate folders within each concentration folder
        for rep_id, replicate_folder in enumerate(os.listdir(conc_folder)):
            replicate_path = os.path.join(conc_folder, replicate_folder)

            # Process only replicate directories
            if os.path.isdir(replicate_path) and replicate_folder.startswith('replicate'):

                # Iterate through position folders within each replicate
                for pos_folder in os.listdir(replicate_path):
                    if pos_folder.startswith('pos'):
                        pos_path = os.path.join(replicate_path, pos_folder, 'single_cell_props.csv')

                        # Load data if CSV file exists
                        if os.path.isfile(pos_path):
                            df = pd.read_csv(pos_path)

                            # Add metadata columns
                            df.insert(1, col_name, conc_label)
                            df.insert(2, 'replicate', rep_id)
                            df.insert(3, 'pos', pos_folder)
                            df.insert(4, 'replicate_name', replicate_folder)

                            # Append to collection
                            all_data.append(df)

    # Combine all dataframes and save to output file
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        combined_df.to_csv(output_dir, index=False)
        print(f"Successfully combined {len(all_data)} datasets")
        print(f"Total rows: {len(combined_df)}")
        print(f"Output saved to: {output_dir}")

        # Display summary
        print(combined_df.columns)
        print(f"{col_name} concentrations found: {sorted(combined_df[col_name].unique())}")
        print(combined_df['replicate_name'].unique())
        print(combined_df['replicate'].unique())

    else:
        print("Warning: No data files found")

Successfully combined 118 datasets
Total rows: 445034
Output saved to: hqno_calibration_microscopy.csv
Index(['label', 'HQNO', 'replicate', 'pos', 'replicate_name', 'area',
       'min_row', 'min_col', 'max_row', 'max_col', 'intensity_max',
       'intensity_mean', 'intensity_min', 'minor_axis_length',
       'major_axis_length', 'y', 'x', 'intensity_gfp', 'intensity_raw_gfp',
       'frame_number', 'intensity_mcherry', 'intensity_raw_mcherry'],
      dtype='str')
HQNO concentrations found: ['HQNO-0ng', 'HQNO-10ng', 'HQNO-160ng', 'HQNO-20ng', 'HQNO-2500ng', 'HQNO-320ng', 'HQNO-40ng', 'HQNO-80ng']
<ArrowStringArray>
['replicate_1', 'replicate_2']
Length: 2, dtype: str
[0 1]
Successfully combined 102 datasets
Total rows: 339071
Output saved to: rhl_calibration_microscopy.csv
Index(['label', 'RHL', 'replicate', 'pos', 'replicate_name', 'area', 'min_row',
       'min_col', 'max_row', 'max_col', 'intensity_max', 'intensity_mean',
       'intensity_min', 'minor_axis_length', 'major_axis_leng